In [67]:
import pandas as pd
import numpy as np
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.model_selection import train_test_split
from sklearn.naive_bayes import MultinomialNB
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.metrics import accuracy_score, roc_auc_score, classification_report, confusion_matrix, f1_score
from sklearn.svm import LinearSVC
from sklearn.model_selection import GridSearchCV
import sklearn
from copy import deepcopy

In [51]:
df = pd.read_json("D:\IT\Python\pytorch\email_classification_project\data\emails.json")

<>:1: SyntaxWarning: invalid escape sequence '\I'
<>:1: SyntaxWarning: invalid escape sequence '\I'
C:\Users\bben2\AppData\Local\Temp\ipykernel_13372\860211338.py:1: SyntaxWarning: invalid escape sequence '\I'
  df = pd.read_json("D:\IT\Python\pytorch\email_classification_project\data\emails.json")


In [52]:
df.head(5)

,text,label,priority,intent
0,Could you schedule a meeting with the client o...,1,1,5
1,I need your approval for the approval form.,1,2,1
2,Your session will expire in 15 minutes.,2,1,3
3,Your session will expire in 15 minutes.,2,1,3
4,Congratulations! You have been selected for a ...,0,1,3


In [53]:
df['label'].value_counts(), df['priority'].value_counts(), df['intent'].value_counts()

(label
 1    150
 2    150
 0    150
 Name: count, dtype: int64,
 priority
 1    219
 0    152
 2     79
 Name: count, dtype: int64,
 intent
 3    151
 4     78
 0     71
 2     54
 5     52
 1     44
 Name: count, dtype: int64)

In [54]:
df.isna().sum()[df.isna().sum()>0]

Series([], dtype: int64)

In [55]:
X = df['text']
y = df.drop('text', axis=1)

In [56]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

In [57]:
pipe = Pipeline([
    ('vect', TfidfVectorizer()),
    ('model', LogisticRegression(max_iter=200))])

In [58]:
target_tasks = ["label", "priority", "intent"]

In [59]:
models = {}

In [69]:
param_grid = {
    "vect__ngram_range": [(1,1), (1,2)],
    "vect__min_df": [1, 3, 5],
    "model__C": [0.1, 1, 3],
    "model__solver": ["lbfgs"]
}

In [75]:
for task in target_tasks:
    y_train_task = y_train[task]
    y_test_task = y_test[task]

    grid = GridSearchCV(
        estimator=deepcopy(pipe),
        param_grid=param_grid,
        scoring="f1_weighted",
        cv=3,
        n_jobs=-1,
        verbose=1
    )
    grid.fit(X_train, y_train_task)
    best_model = grid.best_estimator_
    
    models[task] = best_model

    preds = best_model.predict(X_test)
    
    acc = accuracy_score(y_test_task, preds)

    f1 = f1_score(y_test_task, preds, average="weighted")


    print(f"Accuracy {acc}, f1 score {f1} for the task {task}")
    print(classification_report(y_test_task, preds, zero_division=0))

Fitting 3 folds for each of 18 candidates, totalling 54 fits
Accuracy 1.0, f1 score 1.0 for the task label
              precision    recall  f1-score   support

           0       1.00      1.00      1.00        29
           1       1.00      1.00      1.00        33
           2       1.00      1.00      1.00        28

    accuracy                           1.00        90
   macro avg       1.00      1.00      1.00        90
weighted avg       1.00      1.00      1.00        90

Fitting 3 folds for each of 18 candidates, totalling 54 fits
Accuracy 0.4888888888888889, f1 score 0.49552139037433157 for the task priority
              precision    recall  f1-score   support

           0       0.57      0.47      0.52        36
           1       0.38      0.49      0.42        35
           2       0.67      0.53      0.59        19

    accuracy                           0.49        90
   macro avg       0.54      0.49      0.51        90
weighted avg       0.51      0.49      0.50  

In [64]:
y_train_task.shape

(360,)